# FinSight-RAG — Phase 3 (Final System)
**A Source-Grounded Question-Answering Assistant for Corporate Financial Filings**
Lab 9 PSIS · Activity 1 (RAG-Based Domain Assistant) · Akash Goyal (I025) · Karan Lokhande (I039) · Meer Magia (I040)

**Before you start:** add your Gemini key in Colab: click the 🔑 **Secrets** icon in the left sidebar → *Add new secret* →
name `GOOGLE_API_KEY`, value = your key from Google AI Studio → turn **Notebook access** on. Then **Runtime → Run all**.
The key is read from Colab Secrets and is never printed or saved to the repository.

What this notebook does:
1. Clones the repo, installs the pinned packages and downloads the NIKE FY2023 10-K (107 pages).
2. Runs the full Phase 3 evaluation: semantic vs keyword vs **hybrid** retrieval, answers from Gemini for Q1–Q7
   (Phase 2 baseline vs Phase 3 final system), automatic checks, and the multi-turn follow-up test.
3. Prints every answer next to the full text of the passage it cites, for the manual grounding review.
4. Live demo cells (used in the video) and the test suite.

## 1. Setup

In [ ]:
%cd /content
!test -d finsight-rag-I025-39-40 || git clone -q https://github.com/akashgoyalll/finsight-rag-I025-39-40.git
%cd /content/finsight-rag-I025-39-40
!git pull -q
# If the Phase 3 code is not on GitHub yet: drag finsight-rag-phase3.zip into the Files panel first
# (any folder under /content; a renamed download such as "finsight-rag-phase3 (1).zip" is fine).
import glob, os, subprocess
zips = sorted(glob.glob("/content/**/finsight-rag-phase3*.zip", recursive=True), key=os.path.getmtime)
if zips:
    subprocess.run(["unzip", "-o", "-q", zips[-1], "-d", "."], check=True)
    print("Applied Phase 3 files from", zips[-1])
!git log --oneline -3
if not os.path.exists("scripts/run_phase3.py"):
    raise SystemExit("STOP: Phase 3 code missing. Push it to GitHub, or upload finsight-rag-phase3.zip in the Files panel, then re-run this cell.")
print("Phase 3 code present")
!pip install -q -r requirements.txt
!bash scripts/get_data.sh
!ls -la data

/content
/content/finsight-rag-I025-39-40
c49faea (HEAD -> main, origin/main, origin/HEAD) Update
f3ce32d Update README.md
2db9731 Update README.md
Phase 3 code present
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Gemini API key and model

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API key: ")
# Free-tier Gemini model (Google AI Studio). If you get repeated 429 (rate-limit) errors, switch to
# "google_genai:gemini-3.5-flash-lite" and re-run from here.
os.environ["LLM_MODEL"] = "google_genai:gemini-3.5-flash"
print("Key loaded:", bool(os.environ.get("GOOGLE_API_KEY")), "| model:", os.environ["LLM_MODEL"])

Gemini API key: ··········
Key loaded: True | model: google_genai:gemini-3.5-flash


## 3. Full Phase 3 evaluation
Retrieval comparison for Q1–Q6 (no LLM), then Gemini answers for Q1–Q7 with the **Phase 2 baseline** (semantic, top-4)
and the **Phase 3 system** (hybrid semantic + keyword with reciprocal rank fusion, top-6), then the two-turn memory test
with and without follow-up rewriting. Takes about 5–10 minutes (index build + ~20 LLM calls with a 4-second pause between calls).
Results are saved to `outputs/phase3_results.json`, `outputs/phase3_summary.md` and `outputs/phase3_log_colab.txt`.

In [ ]:
import os, subprocess, sys
# Runs the evaluation, streams its output here and saves it to outputs/phase3_log_colab.txt
with open("outputs/phase3_log_colab.txt", "w") as log:
    p = subprocess.Popen([sys.executable, "scripts/run_phase3.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "unauthenticated requests" in line or "it/s]" in line:
            continue
        print(line, end=""); log.write(line)
    p.wait()
if p.returncode != 0 or not os.path.exists("outputs/phase3_summary.md"):
    raise SystemExit(f"STOP: run_phase3.py failed (exit code {p.returncode}). The error is in the last lines printed above.")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Pages 107 | chunks 516 | semantic index = hf | keyword index = tfidf | built in 90.5s

=== Retrieval comparison (Q1-Q6). string = evidence string in retrieved text; page = a ground-truth page retrieved ===
semantic@4  string 4/6  page 5/6  | Q1:H  Q2:m  Q3:H  Q4:H  Q5:m  Q6:H
keyword@4   string 4/6  page 5/6  | Q1:m  Q2:H  Q3:H  Q4:H  Q5:H  Q6:m
semantic@6  string 4/6  page 6/6  | Q1:H  Q2:m  Q3:H  Q4:H  Q5:m  Q6:H
keyword@6   string 4/6  page 5/6  | Q1:m  Q2:H  Q3:H  Q4:H  Q5:H  Q6:m
hybrid@4    string 6/6  page 6/6  | Q1:H  Q2:H  Q3:H  Q4:H  Q5:H  Q6:H
hybrid@6    string 6/6  page 6/6  | Q1:H  Q2:H  Q3:H  Q4:H  Q5:H  Q6:H

LLM: google_genai:gemini-3.5-flash

=== Answers: baseline (Phase 2: sema

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("outputs/phase3_summary.md").read()))

# Phase 3 results

Model: `google_genai:gemini-3.5-flash` · semantic backend: `hf` · 107 pages, 516 chunks · generated 2026-09-30 19:18:43 +0000

## Retrieval (Q1–Q6)

| Retriever | Q1 | Q2 | Q3 | Q4 | Q5 | Q6 | string hits | page hits |
|---|---|---|---|---|---|---|---|---|
| semantic@4 | HIT [36, 89, 31, 32] | miss [4, 86, 6, 36] | HIT [37, 38, 47, 31] | HIT [36, 36, 31, 43] | miss [46, 46, 82, 88] | HIT [39, 39, 39, 32] | 4/6 | 5/6 |
| keyword@4 | miss [42, 43, 41, 40] | HIT [28, 9, 79, 9] | HIT [37, 38, 48, 47] | HIT [36, 31, 41, 40] | HIT [14, 41, 14, 32] | miss [32, 39, 37, 39] | 4/6 | 5/6 |
| semantic@6 | HIT [36, 89, 31, 32, 4, 35] | miss [4, 86, 6, 36, 89, 9] | HIT [37, 38, 47, 31, 38, 47] | HIT [36, 36, 31, 43, 37, 89] | miss [46, 46, 82, 88, 45, 46] | HIT [39, 39, 39, 32, 31, 36] | 4/6 | 6/6 |
| keyword@6 | miss [42, 43, 41, 40, 6, 44] | HIT [28, 9, 79, 9, 78, 96] | HIT [37, 38, 48, 47, 47, 38] | HIT [36, 31, 41, 40, 5, 6] | HIT [14, 41, 14, 32, 13, 45] | miss [32, 39, 37, 39, 87, 31] | 4/6 | 5/6 |
| hybrid@4 | HIT [36, 42, 89, 43] | HIT [4, 28, 86, 9] | HIT [37, 38, 47, 48] | HIT [36, 31, 36, 41] | HIT [46, 14, 46, 41] | HIT [32, 39, 39, 39] | 6/6 | 6/6 |
| hybrid@6 | HIT [36, 42, 89, 43, 31, 41] | HIT [4, 28, 86, 9, 6, 79] | HIT [37, 38, 47, 48, 31, 47] | HIT [36, 31, 36, 41, 43, 40] | HIT [46, 14, 46, 41, 82, 14] | HIT [32, 39, 39, 39, 37] | 6/6 | 6/6 |

## Answers — baseline (Phase 2: semantic@4)

| Q | answer | answerable | cited pages | answer_ok | cited_ok |
|---|---|---|---|---|---|
| Q1 | In fiscal 2023, NIKE, Inc.'s total revenues were $51.2 billion (or $51,217 million). | True | [36, 89, 31, 32] | True | True |
| Q2 | I am sorry, but the provided excerpts do not contain information about the number of employees NIKE had as of May 31, 2023. | False | [] | False | False |
| Q3 | In fiscal 2023, consolidated gross margin decreased 250 basis points to 43.5% compared to 46.0% for fiscal 2022. Meanwhile, consolidated gross profit increased 4% to $22,292 million compared to $21,479 million for fiscal 2022. | True | [37] | True | True |
| Q4 | NIKE Direct revenues grew 14% from $18.7 billion in fiscal 2022 to $21.3 billion in fiscal 2023. On a currency-neutral basis, NIKE Direct revenues increased 20%. | True | [36, 31] | True | True |
| Q5 | NIKE conducts business in various currencies, which subjects it to foreign currency risk that can impact its consolidated results of operations, financial position, and cash flows. The company's foreign currency exposures include:  1. **Product Costs:** Primarily generated through non-functional currency denominated product purchases. Specifically, a weakening U.S. Dollar against foreign currencies within the factory currency exposure indices increases NIKE's U.S. Dollar inventory cost, while a strengthening U.S. Dollar reduces it. 2. **Non-functional currency denominated revenues** 3. **Intercompany revenues** 4. **Demand creation expenses** 5. **Investments in U.S. Dollar denominated available-for-sale debt securities** 6. **Certain other intercompany transactions** | True | [46, 46, 82] | None | True |
| Q6 | To find the EBIT margin for each NIKE Brand geographic segment in fiscal 2023, we divide each segment's EBIT by its Revenues:  1. **Greater China**:    - EBIT: $2,283 million [2]    - Revenues: $7,248 million [1]    - EBIT Margin: $2,283 / $7,248 ≈ 31.5%  2. **Asia Pacific & Latin America**:    - EBIT: $1,932 million [3]    - Revenues: $6,431 million [1]    - EBIT Margin: $1,932 / $6,431 ≈ 30.0%  3. **Europe, Middle East & Africa**:    - EBIT: $3,531 million [2]    - Revenues: $13,418 million [1]    - EBIT Margin: $3,531 / $13,418 ≈ 26.3%  4. **North America**:    - EBIT: $5,454 million [2]    - Revenues: $21,608 million [1]    - EBIT Margin: $5,454 / $21,608 ≈ 25.2%  Therefore, **Greater China** had the highest EBIT margin in fiscal 2023 at approximately 31.5%. | True | [39, 39, 39, 32] | True | True |
| Q7 | I am sorry, but the provided excerpts do not contain information about NIKE's total revenues in fiscal 2025. The data only covers up to fiscal 2023. | False | [] | True | None |

## Answers — final (Phase 3: hybrid@6)

| Q | answer | answerable | cited pages | answer_ok | cited_ok |
|---|---|---|---|---|---|
| Q1 | In fiscal 2023, NIKE, Inc.'s total revenues were $51.2 billion (or $51,217 million). | True | [36, 89, 31] | True | True |
| Q2 | As of May 31, 2023, NIKE had approximately 83,700 employees worldwide, including retail and part-time employees. | True | [9] | True | True |
| Q3 | In fiscal 2023, gross margin decreased 250 basis points to 43.5% compared to 46.0% for fiscal 2022. This decrease was primarily driven by:  - Higher NIKE Brand product costs, on a wholesale equivalent basis, primarily due to higher input costs and elevated inbound freight and logistics costs as well as product mix; - Lower margin in the NIKE Direct business, driven by higher promotional activity to liquidate inventory in the current period compared to lower promotional activity in the prior period;  - Unfavorable changes in net foreign currency exchange rates, including hedges; and - Lower off-price margin, on a wholesale equivalent basis. | True | [37] | True | True |
| Q4 | In fiscal 2023, NIKE Direct revenues increased 14% (or 20% on a currency-neutral basis) from $18.7 billion in fiscal 2022 to $21.3 billion. | True | [36, 31] | True | True |
| Q5 | According to the provided filings, NIKE faces several risks from fluctuations in foreign currency exchange rates:  1. **Financial Impact:** Fluctuations could result in lower revenues, higher costs, decreased margins and earnings, and have an adverse effect on results of operations and financial condition [2, 6].  2. **Translation Effects:** Because international revenues and expenses are derived from sales and operations in foreign currencies, a weakening of foreign currencies relative to the U.S. Dollar adversely affects the U.S. Dollar value of the Company's foreign currency-denominated sales and earnings [6]. 3. **Product and Inventory Costs:** NIKE faces transactional foreign currency risks, primarily related to product costs from non-functional currency denominated product purchases [3, 5]. For example, a weakening U.S. Dollar against foreign currencies in the factory currency exposure index increases NIKE's inventory cost [1]. 4. **Supply Chain Disruption:** Currency exchange rate fluctuations can disrupt the business of independent manufacturers that produce NIKE's products by making their purchases of raw materials more expensive and more difficult to finance [6]. | True | [46, 14, 46, 82, 14] | None | True |
| Q6 | Based on the provided excerpts, the EBIT margins for the NIKE Brand geographic segments in fiscal 2023 are calculated as follows:  * **Greater China**: EBIT of $2,283 million divided by Revenues of $7,248 million equals approximately **31.50%**. * **Asia Pacific & Latin America**: EBIT of $1,932 million divided by Revenues of $6,431 million equals approximately **30.04%**. * **Europe, Middle East & Africa**: EBIT of $3,531 million divided by Revenues of $13,418 million equals approximately **26.31%**. * **North America**: EBIT of $5,454 million divided by Revenues of $21,608 million equals approximately **25.24%**.  Therefore, **Greater China** had the highest EBIT margin among the NIKE Brand geographic segments in fiscal 2023. | True | [39, 39, 39] | True | True |
| Q7 | I am sorry, but the provided excerpts do not contain information about NIKE's total revenues in fiscal 2025, as the data only goes up to fiscal 2023. | False | [] | True | None |

## Multi-turn follow-up (expected: 16% currency-neutral)

- **without question rewriting (Phase 2)** — retrieval query: “How much did that grow on a currency-neutral basis?” → On a currency-neutral basis, NIKE, Inc. Revenues increased 16% in fiscal 2023 compared to fiscal 2022. (answer_ok=True, history messages=4)
- **with question rewriting (Phase 3)** — retrieval query: “How much did NIKE, Inc.'s total revenues grow on a currency-neutral basis in fiscal 2023?” → In fiscal 2023, NIKE, Inc.'s revenues increased 16% on a currency-neutral basis compared to fiscal 2022. (answer_ok=True, history messages=4)


## 4. Manual grounding review
For each answer of the final system: the answer, then the **full text of every chunk it cites** (document, page).
Grounded = every figure/claim in the answer appears in (or is directly computed from) the cited text.

In [ ]:
import json, sys, textwrap
sys.path.insert(0, "src")
from finsight.indexing import load_pages, chunk
from finsight.hybrid import chunk_key
chunks = chunk(load_pages(["data/nke-10k-2023.pdf"]))
by_key = {chunk_key(c): c for c in chunks}
res = json.load(open("outputs/phase3_results.json"))
final = [k for k in res["answers"] if k.startswith("final")][0]
for r in res["answers"][final]:
    print("=" * 100)
    print(f"{r['id']}  {r['question']}\nTRUTH : {r['ground_truth']}\nANSWER: {r['answer'] or r.get('error')}")
    print(f"answerable={r['answerable']}  auto answer_ok={r['answer_ok']}  cited_ok={r['cited_ok']}")
    for s in r["sources"]:
        c = by_key.get((s["source"], s["page"], s["start_index"], s["passage"][:50]))
        print(f"  --- cited [{s['excerpt']}] {s['source']} p.{s['page']} ---")
        print(textwrap.indent(textwrap.fill((c.page_content if c else s["passage"]), 110), "    "))

Q1  What were NIKE, Inc.'s total revenues in fiscal 2023?
TRUTH : $51.2 billion ($51,217 million), +10% reported / +16% currency-neutral
ANSWER: In fiscal 2023, NIKE, Inc.'s total revenues were $51.2 billion (or $51,217 million).
answerable=True  auto answer_ok=True  cited_ok=True
  --- cited [1] nke-10k-2023.pdf p.36 ---
    Table of Contents FISCAL 2023 NIKE BRAND REVENUE HIGHLIGHTSThe following tables present NIKE Brand revenues
    disaggregated by reportable operating segment, distribution channel and major product line: FISCAL 2023
    COMPARED TO FISCAL 2022 • NIKE, Inc. Revenues were $51.2 billion in fiscal 2023, which increased 10% and 16%
    compared to fiscal 2022 on a reported and currency-neutral basis, respectively. The increase was due to higher
    revenues in North America, Europe, Middle East & Africa ("EMEA"), APLA and Greater China, which contributed
    approximately 7, 6, 2 and 1 percentage points to NIKE, Inc. Revenues, respectively. • NIKE Brand revenues,
    w

## 5. Live demo (used in the video)
Builds the final system in memory (hybrid retrieval + Gemini + memory) and answers questions with page citations.

In [ ]:
import warnings
from langchain_core._api import LangChainDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
from finsight.indexing import index_chunks
from finsight.hybrid import build_hybrid_retriever
from finsight.chain import build_chain, with_memory
from finsight.llm import get_llm

retriever = build_hybrid_retriever(index_chunks(chunks, kind="hf"), index_chunks(chunks, kind="tfidf"), fetch_k=4, k=6)
chat = with_memory(build_chain(retriever, get_llm()))

def ask(question, session="demo"):
    out = chat.invoke({"question": question}, config={"configurable": {"session_id": session}})
    print(f"Q: {question}")
    if out["retrieval_query"] != question:
        print(f"   (rewritten for retrieval: {out['retrieval_query']})")
    print(f"A: {out['answer']}\n   answerable = {out['answerable']}")
    for s in out["sources"]:
        print(f"   source [{s['excerpt']}]: {s['source']}, page {s['page']}  |  {s['passage'][:120].replace(chr(10), ' ')}...")
    print(f"   retrieved pages: {out['retrieved_pages']}\n")
    return out

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
_ = ask("How many employees did NIKE have as of May 31, 2023?", session="demo-1")

GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 36.381386675s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [ ]:
_ = ask("What were NIKE, Inc.'s total revenues in fiscal 2023?", session="demo-2")
_ = ask("How much did that grow on a currency-neutral basis?", session="demo-2")   # follow-up uses memory

In [ ]:
_ = ask("Which NIKE Brand geographic segment had the highest EBIT margin in fiscal 2023?", session="demo-3")  # weak case

In [ ]:
_ = ask("What were NIKE's total revenues in fiscal 2025?", session="demo-4")  # not in the filing -> should refuse

In [ ]:
# Try your own question
_ = ask("What was NIKE's diluted earnings per share in fiscal 2023?", session="demo-5")

## 6. Test suite (offline, no API calls)

In [ ]:
!python -m pytest tests/ -v -p no:cacheprovider -W ignore::DeprecationWarning 2>&1 | tee outputs/phase3_pytest_log_colab.txt | tail -16

## 7. Download the results
Downloads `phase3_outputs.zip` (results JSON, summary, logs). Send it to the report owner / commit it to `outputs/`.

In [ ]:
!cd outputs && zip -q -j ../phase3_outputs.zip phase3_results.json phase3_summary.md phase3_log_colab.txt phase3_pytest_log_colab.txt
from google.colab import files
files.download("phase3_outputs.zip")